In [1]:
!pip install pyserial

In [2]:
import serial, time
!pip install pyserial

In [169]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [170]:
print(serial)

<module 'serial' from 'C:\\Users\\boome\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [171]:
print(serial.__file__)

C:\Users\boome\anaconda3\Lib\site-packages\serial\__init__.py


In [172]:
print(serial.__version__)

3.5


In [173]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [174]:
baudrate = 115200

In [175]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [177]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [178]:
ser.in_waiting

0

In [181]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [182]:
read_all(ser)

'dual servo control over serial\n'

In [183]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [184]:
read_one_line(ser)

''

In [185]:
read_all(ser)

''

In [186]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [187]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

# Break an integer into two bytes

In [189]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [208]:
# inputs from user
xll = 10 # x origin
yll = 33 # y origin

w =  -8  # width (dont worry about the negative)
h = 8   # height

K = 10  # number of steps per side
N = (3*K) #refining the robots movement

In [209]:
#define step size
dx = w/N
dy = h/N

#generate bottom coordinants
x_bottom = np.linspace(xll, xll+w-dx, N)  
y_bottom = np.full(N, yll)

#generate right coordinants
x_right = np.full(N, xll + w)  
y_right = np.linspace(yll, yll+h-dy, N)

#generate top coordinants
x_top = np.linspace(xll+w, xll+dx, N)  
y_top = np.full(N,yll+h)

#generate left coordinants
x_left = np.full(N+2, xll)  
y_left = np.linspace(yll+h, yll, N+1)
y_left = np.append(y_left, yll)

#combine bottom, right, top, left into one array
x_path = np.concatenate((x_bottom, x_right, x_top, x_left), axis=0) 
y_path = np.concatenate((y_bottom, y_right, y_top, y_left), axis=0) 

#combine x and y into one array
tip_path = np.column_stack((x_path, y_path))
tip_path

array([[10.        , 33.        ],
       [ 9.73333333, 33.        ],
       [ 9.46666667, 33.        ],
       [ 9.2       , 33.        ],
       [ 8.93333333, 33.        ],
       [ 8.66666667, 33.        ],
       [ 8.4       , 33.        ],
       [ 8.13333333, 33.        ],
       [ 7.86666667, 33.        ],
       [ 7.6       , 33.        ],
       [ 7.33333333, 33.        ],
       [ 7.06666667, 33.        ],
       [ 6.8       , 33.        ],
       [ 6.53333333, 33.        ],
       [ 6.26666667, 33.        ],
       [ 6.        , 33.        ],
       [ 5.73333333, 33.        ],
       [ 5.46666667, 33.        ],
       [ 5.2       , 33.        ],
       [ 4.93333333, 33.        ],
       [ 4.66666667, 33.        ],
       [ 4.4       , 33.        ],
       [ 4.13333333, 33.        ],
       [ 3.86666667, 33.        ],
       [ 3.6       , 33.        ],
       [ 3.33333333, 33.        ],
       [ 3.06666667, 33.        ],
       [ 2.8       , 33.        ],
       [ 2.53333333,

In [210]:
#define link lengths
l1 = 24 # base link
l2 = 21.5 # tip link

#distance from origin to tip
r_squared = tip_path[:,0]**2 + tip_path[:,1]**2

#law of cos for angle between links
alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
print(alpha_temp)
alpha = np.arccos(alpha_temp)

print(alpha*rtd)

#vertical angle theorem for theta 2
theta2 = 180 - alpha*rtd

#triangle in link1 co-ordinant system for psi
psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))*rtd

#angle of r to x-axis
beta = np.arctan2(tip_path[:,1],tip_path[:,0])*rtd

#difference in beta and psi is theta 1
theta1 = beta - psi

# check that values are possible before continuing
Check = 0
for i in range(len(theta1)):
    
    if theta2[i] < 0:
        theta2[i] = theta2[i]+360
for i in range(len(theta1)):
    if theta1[i] < 0 or theta1[i] >180:
        Check = 1
    if (theta2[i] > 90 and theta2[i] < 270) or (theta2[i] < 0):
        Check = 1
if Check == 0:
    print("Correct")
if Check == 1:
    print("Wrong")
print("\ntheta 1:\n",theta1)
print("theta 2:\n",theta2)

[-0.14607558 -0.14097653 -0.13601529 -0.13119186 -0.12650624 -0.12195844
 -0.11754845 -0.11327627 -0.1091419  -0.10514535 -0.10128661 -0.09756568
 -0.09398256 -0.09053725 -0.08722976 -0.08406008 -0.08102821 -0.07813415
 -0.07537791 -0.07275947 -0.07027885 -0.06793605 -0.06573105 -0.06366387
 -0.0617345  -0.05994294 -0.05828919 -0.05677326 -0.05539513 -0.05415482
 -0.05305233 -0.0701755  -0.08743648 -0.10483527 -0.12237188 -0.1400463
 -0.15785853 -0.17580857 -0.19389643 -0.21212209 -0.23048557 -0.24898686
 -0.26762597 -0.28640289 -0.30531761 -0.32437016 -0.34356051 -0.36288867
 -0.38235465 -0.40195844 -0.42170004 -0.44157946 -0.46159668 -0.48175172
 -0.50204457 -0.52247524 -0.54304371 -0.56375    -0.5845941  -0.60557601
 -0.62669574 -0.62779823 -0.62903854 -0.63041667 -0.6319326  -0.63358635
 -0.63537791 -0.63730728 -0.63937446 -0.64157946 -0.64392227 -0.64640289
 -0.64902132 -0.65177756 -0.65467162 -0.65770349 -0.66087317 -0.66418066
 -0.66762597 -0.67120909 -0.67493002 -0.67878876 -0.

In [211]:
#convert angle into arduino code
theta_min = 0     # minimum angle
theta_max = 180   # maximum angle
min_new = 1000    # minimum servo value
max_new = 2000    # maximum servo value

#linear interpolate for the first theta value
myint = min_new + ((theta1-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print(myint)

#linear interpolate for the second theta value; 180 and 90 included to account for robots home position
myint2 = min_new + (((180-(90+theta2))-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print('\n',myint2)

[1194.75975322 1196.37946821 1198.03045577 1199.71243294 1201.42511528
 1203.16821669 1204.94144924 1206.74452302 1208.57714601 1210.43902396
 1212.32986031 1214.24935609 1216.19720988 1218.17311777 1220.17677334
 1222.20786764 1224.26608918 1226.35112403 1228.46265576 1230.60036558
 1232.76393235 1234.95303271 1237.16734115 1239.40653012 1241.67027017
 1243.95823008 1246.270077   1248.60547663 1250.96409336 1253.34559049
 1255.74963037 1258.3522794  1260.97969914 1263.632762   1266.3123916
 1269.01956691 1271.75532694 1274.52077586 1277.31708879 1280.14551821
 1283.00740115 1285.90416732 1288.83734823 1291.80858749 1294.81965256
 1297.87244808 1300.96903115 1304.11162882 1307.30265833 1310.54475051
 1313.84077699 1317.19388215 1320.60752065 1324.08550187 1327.63204296
 1331.25183246 1334.95010737 1338.73274715 1342.60638954 1346.57857463
 1350.65792629 1348.80331071 1346.97673619 1345.17848209 1343.40884008
 1341.66811453 1339.95662278 1338.27469564 1336.62267779 1335.00092839
 1333.4

In [212]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [213]:
#define arrays to hold the four bytes
byte1 = np.zeros(len(theta1), dtype=int)
byte2 = np.zeros(len(theta1), dtype=int)
byte3 = np.zeros(len(theta1), dtype=int)
byte4 = np.zeros(len(theta1), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(theta1)):
    byte1[i], byte2[i] = break_into_two(myint[i])
    byte3[i], byte4[i] = break_into_two(myint2[i])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4)

[4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5
 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 5 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4] 

 [170 172 174 175 177 179 180 182 184 186 188 190 192 194 196 198 200 202
 204 206 208 210 213 215 217 219 222 224 226 229 231 234 236 239 242 245
 247 250 253   0   3   5   8  11  14  17  20  24  27  30  33  37  40  44
  47  51  54  58  62  66  70  68  66  65  63  61  59  58  56  55  53  51
  50  48  47  45  44  43  41  40  39  38  36  35  34  33  32  31  30  29
  28  23  19  14   9   5   1 252 248 244 240 236 232 229 225 221 218 214
 210 207 203 200 197 193 190 187 183 180 177 173 170 170] 


 [4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 3 3 3 3 3 3 3 3 3 3 3 3 3 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
 4 4 4 4 4 4 4 4 4 4 4]

In [214]:
# Send all path points to both servos
x = 1
for i in range(len(theta1)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)

    # read back confirmation from Arduino
    #line1 = read_one_line(ser)  # servo 1 bytes echo
   # line2 = read_one_line(ser)  # servo 1 int echo
    #line3 = read_one_line(ser)  # servo 2 bytes echo
    #line4 = read_one_line(ser)  # servo 2 int echo
    #print(f"Step {i}: servo1={line2}  servo2={line4}")

   # print('\ntheta1:',theta1[i],'\ntheta2:',theta2[i],'\n','\n\n')
 
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            print(response)
            if response == "Ready":
                break
            else: 
                line1 = read_one_line(ser)  # servo 1 bytes echo
                line2 = read_one_line(ser)  # servo 1 int echo
                print(f"Step {i-1}: servo1={line1}  servo2={line2}")
                line3 = read_one_line(ser)  # servo 1 bytes echo
                line4 = read_one_line(ser)  # servo 1 int echo
                                              # servo 2 int echo
                print(f"Step {i}: servo1={line3}  servo2={line4}") 
            #response = ser.readline().decode('utf-8').strip()
            
                
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move


Step 0: servo1=My int: 1194  servo2=My int2: 1046
Step 1: servo1=My int3: 1196  servo2=My int4: 1045
Ready

Step 2: servo1=My int: 1198  servo2=My int2: 1043
Step 3: servo1=My int3: 1199  servo2=My int4: 1041
Ready

Step 4: servo1=My int: 1201  servo2=My int2: 1040
Step 5: servo1=My int3: 1203  servo2=My int4: 1038
Ready

Step 6: servo1=My int: 1204  servo2=My int2: 1037
Step 7: servo1=My int3: 1206  servo2=My int4: 1036
Ready

Step 8: servo1=My int: 1208  servo2=My int2: 1034
Step 9: servo1=My int3: 1210  servo2=My int4: 1033
Ready

Step 10: servo1=My int: 1212  servo2=My int2: 1032
Step 11: servo1=My int3: 1214  servo2=My int4: 1031
Ready

Step 12: servo1=My int: 1216  servo2=My int2: 1029
Step 13: servo1=My int3: 1218  servo2=My int4: 1028
Ready

Step 14: servo1=My int: 1220  servo2=My int2: 1027
Step 15: servo1=My int3: 1222  servo2=My int4: 1026
Ready

Step 16: servo1=My int: 1224  servo2=My int2: 1025
Step 17: servo1=My int3: 1226  servo2=My int4: 1024
Ready

Step 18: servo1=My 